# Trabalho final
Aluno: Pedro H. M. Tashima

## Introdução
Dois datasets foram utilizados. Ambos foram gerados utilizando a base de dados Sci Expanded do Web Of Science. O tema principal dos dois é internet das coisas e edge computing. Apenas artigos publicados entre 2000 e 2025 foram utilizados.

O primeiro dataset possui 10071 artigos e é mais amplo. O segundo, possui 511 artigos e é focado em replicação de dados.

O objetivo é criar um modelo, utilizando o primeiro dataset e classificar artigos os artigos do segundo em muito relevantes, ou não, utilizando suas keywords, categorias e métricas de citações, acessos, ano, etc.

In [22]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

df = pd.read_csv("../../../CARS007/trabalho-1-bibliometria/data/savedrecs.txt", sep="\t", usecols=["DE", "ID", "CR", "NR", "TC", "Z9", "U1", "U2", "PY", "WC", "SC"])
df_test = pd.read_csv("../../../CARS007/trabalho-2-cloud/data/savedrecs.txt", sep="\t", usecols=["DE", "ID", "CR", "NR", "TC", "Z9", "U1", "U2", "PY", "WC", "SC"])

## Preparação dos dados

### Renomeia colunas

In [23]:
columns = {
    "DE": "author_keywords",
    "ID": "keywords_plus",
    "CR": "cited_references",
    "NR": "cited_references_count",
    "TC": "wos_times_cited",
    "Z9": "total_times_cited",
    "U1": "usage_180d",
    "U2": "usage_since_2013",
    "PY": "year",
    "WC": "categories",
    "SC": "research_areas",
}
df = df.rename(columns=columns)
df_test = df_test.rename(columns=columns)

print(df.describe())
print(df_test.describe())

       cited_references_count  wos_times_cited  total_times_cited  \
count            10071.000000     10071.000000       10071.000000   
mean                48.966041        27.726442          30.577500   
std                 41.440761        92.051192         102.202481   
min                  0.000000         0.000000           0.000000   
25%                 30.000000         2.000000           2.000000   
50%                 39.000000         8.000000           9.000000   
75%                 51.000000        25.000000          27.000000   
max                737.000000      4775.000000        5323.000000   

         usage_180d  usage_since_2013          year  
count  10071.000000      10071.000000  10071.000000  
mean       2.795849         26.961076   2021.983517  
std        5.397122         53.125622      2.173307  
min        0.000000          0.000000   2003.000000  
25%        0.000000          7.000000   2021.000000  
50%        1.000000         14.000000   2022.000000  


### Junta keywords e categorias em uma coluna e transforma o texto

In [24]:
def prepare_keywords(df):
    df["keywords"] = df["author_keywords"] + df["keywords_plus"]
    df["keywords"] = df["keywords"].str.lower()
    df["keywords"] = df["keywords"].fillna("")

    df["categories"] = df["categories"].str.lower()
    df["categories"] = df["categories"].fillna("")

    df["keywords_categories"] = df["categories"] + df["keywords"]

    PONTUACAO = string.punctuation

    def remove_pontuacao(text):
        return text.translate(str.maketrans('', '', PONTUACAO))

    df["keywords_categories"] = df["keywords_categories"].apply(lambda text: remove_pontuacao(text))

    return df

df = prepare_keywords(df)
df_test = prepare_keywords(df_test)

## Transforma os textos criados com base nas keywords e categorias em dados numéricos e em novas features no dataset
Utilizando o método Bag of Words, as keywords e categorias são transformadas em dados numéricos. Nenhum processamento adicional como stemming foi realizado, porque os textos utilizados não são naturais.

In [25]:
vectorizer = CountVectorizer()
X_dense = pd.DataFrame(vectorizer.fit_transform(df["keywords_categories"]).toarray(), columns=vectorizer.get_feature_names_out())

X_dense_test = pd.DataFrame(vectorizer.transform(df_test["keywords_categories"]).toarray(), columns=vectorizer.get_feature_names_out())

### Define um valor mínimo de número de vezes que o artigo foi citado para ser considerado como um artigo de impacto relevante
O valor foi definido arbitrariamente com base em uma análise exploratória dos dados utilizados

In [26]:
y = df["total_times_cited"] >= 20
y_test = df_test["total_times_cited"] >= 20


## Modelagem

### Define as colunas que serão utilizadas para treinamento e adiciona colunas criadas com as keywords

In [19]:
X = df[["usage_since_2013", "usage_180d", "cited_references_count", "year"]]
X_test = df_test[["usage_since_2013", "usage_180d", "cited_references_count", "year"]]

X = pd.concat([X, X_dense], axis=1).fillna(0)
X_test = pd.concat([X_test, X_dense_test], axis=1).fillna(0)

X

,usage_since_2013,usage_180d,cited_references_count,year,040,0big,0cyberphysical,0internet,0networking,1000,...,zkpchallenges,zkpdifferential,zkpexchange,zkpkey,zksnarks,zo,zones,zoologyprecision,zooming,zt
0,24,0,50,2019,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,44,3,55,2021,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,10,0,29,2020,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,54,2,41,2018,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,13,0,65,2022,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10066,6,0,36,2024,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10067,7,0,47,2019,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10068,4,4,46,2025,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10069,20,6,31,2021,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Realiza um grid search para encontrar os melhores parametros para o modelo de regressão logistica

In [ ]:
param_grid = {
    'C': np.logspace(-4, 4, 20),
    'penalty': ['l1', 'l2']
}

model = LogisticRegression(max_iter=2000, solver='liblinear')

grid_search = GridSearchCV(model, param_grid, cv=5)
grid_search.fit(X, y)

print("Best parameters: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

Best parameters:  {'C': 0.03359818286283781, 'penalty': 'l2'}
Best cross-validation score:  0.8134237015827175


### Treina o modelo e o utiliza no dataset de testes

In [27]:
model_test = LogisticRegression(max_iter=2000, solver='liblinear', C=0.0335, penalty='l2')
model_test.fit(X, y)

model_test.score(X_test, y_test)

0.8512720156555773

## Resultados e discussão

Os resultados obtidos foram muito positivos, com 85% de acurácia quando utilizado no dataset de testes. Porém, a possibilidade do dataset de testes compartilhar entradas com o dataset de treinamento não foi considerada no início e isso pode ter influenciado nos resultados positivos. Pela restrição de tempo, não foi possível gerar outro dataset e garantir que ele não compartilha entradas com o dataset de treinamento. Entretanto, a validação 5-fold realizada durante os testes com o modelo para encontrar melhores parâmetros, mostrou que o modelo produz resultados muito superiores aos obtidos aleatoriamente.